<a href="https://colab.research.google.com/github/paolobalasso/DataScienceCourse/blob/main/notebooks/04_portfolio_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 04 · Portfolio Optimization — Exploratory & Didactic

**Data Science · Università Cattolica del Sacro Cuore, Milano**
Instructor: *Paolo Balasso*

---

## What you will explore

A guided exploration of portfolio construction with real market data, focused on **understanding** rather than production deployment.

1. **Market data** — pull 8 ETFs across asset classes, inspect, visualise
2. **Returns** — daily and monthly returns, distributions, fat tails
3. **Correlations** — diversification regimes
4. **Macro variables** — FRED data (GDP, CPI, Unemployment, 10y yield, USREC), YoY transformations
5. **Investment Clock** — 4-phase regime classification (Reflection / Recovery / Overheat / Stagflation)
6. **Conditional analysis** — what worked in each regime?
7. **Simple portfolios** — Equal-Weight, Min-Vol, Max-Sharpe, Permanent (Browne), All-Weather (Dalio)
8. **Regime-aware tilt** — overlay regime info on a baseline allocation
9. **Caveats** — look-ahead bias, in-sample optimism, small samples, publication lag

## Approach

Short, readable functions. Lots of plots. Explicit data-science hygiene checks (shapes, NaN, duplicates, train/test awareness).
**Not** a production backtest. The goal is *content sedimentation*: making the concepts stick.

> *Past performance is not indicative of future results. This notebook is for educational purposes only. Nothing here constitutes investment advice.*

---

## 0 · Setup

In [ ]:
# Colab-friendly install (silent)
%pip install -q yfinance pandas_datareader scipy matplotlib seaborn

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import yfinance as yf
import pandas_datareader.data as web
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize

# v7 palette
NAVY, GOLD, INK, MUTED = '#1B263B', '#D4A017', '#0F141E', '#556070'
GREEN, RED = '#2D6E3E', '#B32D2D'
sns.set_style('whitegrid')
plt.rcParams.update({'font.family': 'DejaVu Sans',
                     'axes.edgecolor': MUTED, 'axes.labelcolor': INK})

TRADING_DAYS = 252
rf_annual = 0.04        # approx 10y Treasury yield avg over the period
np.random.seed(42)
print('Setup complete.')

## 1 · The investable universe

We pick **8 ETFs** across the main asset classes — small enough to inspect manually, broad enough to show diversification.

In [ ]:
# 8-ETF universe — diversified across asset classes
TICKERS = {
    'SPY': 'US Large-Cap (S&P 500)',
    'QQQ': 'US Tech (Nasdaq 100)',
    'IWM': 'US Small-Cap (Russell 2000)',
    'EFA': 'Developed Intl ex-US',
    'TLT': 'US Long Treasury',
    'IEF': 'US Intermediate Treasury',
    'GLD': 'Gold',
    'DBC': 'Broad Commodities',
}
START, END = '2014-01-01', '2024-12-31'
print(f'{len(TICKERS)} ETFs across 5 asset classes')
for t, name in TICKERS.items():
    print(f'  {t:<5}  {name}')

In [ ]:
# Download adjusted closes via yfinance
raw = yf.download(list(TICKERS.keys()), start=START, end=END,
                  auto_adjust=True, progress=False)['Close']
print('Raw shape (before cleaning):', raw.shape)
print('Missing values per ticker:')
print(raw.isna().sum())

In [ ]:
# DS-hygiene: forward-fill small gaps (holidays), drop leading/trailing NaN, dedup
prices = raw.dropna(how='all').ffill().dropna()
assert not prices.index.duplicated().any(), 'duplicate dates!'
assert not prices.isna().any().any(),       'residual NaN!'
print(f'Clean price matrix: {prices.shape[0]} days x {prices.shape[1]} assets')
print(f'From {prices.index.min().date()} to {prices.index.max().date()}')
prices.head()

In [ ]:
# Sanity-check: cumulative wealth (start=1) on log scale
norm = prices / prices.iloc[0]
fig, ax = plt.subplots(figsize=(11, 5))
for t in norm.columns:
    ax.plot(norm.index, norm[t], lw=1.1, alpha=0.85, label=t)
ax.set_yscale('log')
ax.set_title('Cumulative wealth, 2014-2024 (log scale, start=1)',
             fontweight='bold', color=INK)
ax.set_ylabel('Wealth multiple')
ax.legend(loc='upper left', ncol=4, fontsize=9)
plt.tight_layout(); plt.show()

## 2 · From prices to returns

Two definitions students need to internalise:

- **Simple return** $r_t = P_t / P_{t-1} - 1$ — *additive across assets* in a portfolio
- **Log return**    $\ell_t = \ln(P_t / P_{t-1})$ — *additive across time*, symmetric around zero

We use simple returns for portfolio math, log returns only for inspection.

In [ ]:
# Daily simple and log returns
rets = prices.pct_change().dropna()
log_rets = np.log(prices / prices.shift(1)).dropna()
print('Daily simple returns:', rets.shape)
print('Any NaN?',              rets.isna().any().any())
print('Any duplicate dates?',  rets.index.duplicated().any())
rets.describe().round(4)

In [ ]:
# Small-multiples histogram: daily return distribution per asset
fig, axes = plt.subplots(2, 4, figsize=(13, 6), sharex=True)
for ax, t in zip(axes.flat, rets.columns):
    ax.hist(rets[t]*100, bins=60, color=NAVY, alpha=0.75,
            edgecolor='white', linewidth=0.3)
    ax.axvline(0, color=GOLD, ls='--', lw=0.8)
    ax.set_title(t, fontweight='bold', color=INK, fontsize=10)
    ax.set_xlabel('Daily return (%)')
plt.suptitle('Daily return distributions — note the fat tails',
             fontweight='bold', color=INK, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Summary stats: annualised return, vol, Sharpe, skew, kurtosis, max DD
rf_daily = rf_annual / TRADING_DAYS
summary = pd.DataFrame({
    'Ann. Return': rets.mean() * TRADING_DAYS,
    'Ann. Vol':    rets.std() * np.sqrt(TRADING_DAYS),
    'Sharpe':      (rets.mean() - rf_daily) / rets.std() * np.sqrt(TRADING_DAYS),
    'Skew':        rets.skew(),
    'Kurtosis':    rets.kurtosis(),
    'Max DD':      ((prices / prices.cummax()) - 1).min(),
}).round(3).sort_values('Sharpe', ascending=False)
summary

**Read-outs:**
- Excess kurtosis > 0 for **all** assets — daily returns are **not** Gaussian.
- Equities show negative skew (more left-tail surprises).
- Bonds offer a competitive Sharpe at lower vol — the classical diversification benefit.
- Max drawdowns are non-trivial for everything: diversification ≠ free lunch.

## 3 · Correlations — the diversification story

Correlations are **not** constants. Below: full-sample heatmap, then a rolling-window view of the famous SPY-TLT pair.

In [ ]:
# Full-sample correlation heatmap
corr = rets.corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, cbar_kws={'shrink': 0.7}, ax=ax)
ax.set_title('Daily return correlations — full sample',
             fontweight='bold', color=INK)
plt.tight_layout(); plt.show()

In [ ]:
# Rolling 60-day correlation: equities vs long bonds
roll = rets['SPY'].rolling(60).corr(rets['TLT'])
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(roll.index, roll, color=NAVY, lw=1.2)
ax.axhline(0, color=MUTED, lw=0.7, ls='--')
ax.fill_between(roll.index, roll, 0, where=(roll < 0),
                color=GREEN, alpha=0.25, label='Diversifying')
ax.fill_between(roll.index, roll, 0, where=(roll >= 0),
                color=RED, alpha=0.25, label='Co-moving')
ax.set_title('Rolling 60-day correlation: SPY vs TLT',
             fontweight='bold', color=INK)
ax.legend(loc='upper left')
plt.tight_layout(); plt.show()
print('\n2022: correlation flipped positive — both equities and long bonds fell together.')
print('Any portfolio built on a stable equity-bond correlation got hurt.')

## 4 · Macro variables from FRED

Markets do not float in a vacuum. We pull four macro series from **FRED** (Federal Reserve Economic Data), the same source used by central banks and institutional research:

| Series      | What it is                              | Frequency | Transformation |
|-------------|------------------------------------------|-----------|----------------|
| `GDPC1`    | Real GDP                                  | Quarterly | YoY % change   |
| `CPILFESL` | Core CPI (excludes food & energy)        | Monthly   | YoY % change   |
| `UNRATE`   | Civilian unemployment rate                | Monthly   | Level          |
| `DGS10`    | 10-year Treasury yield                    | Daily     | Monthly avg    |
| `USREC`    | NBER recession indicator (0 / 1)         | Monthly   | Used as overlay|

The **YoY transformation** is critical: raw GDP grows ~5% in nominal terms every year just from price inflation; the YoY pct change is what tells you whether the economy is actually expanding or contracting.

*Inspiration: Paolo Balasso's [YoutubeTrading/EconomicRegime](https://github.com/paolobalasso/YoutubeTrading/blob/main/EconomicRegime.ipynb).*

In [ ]:
# Pull macro series from FRED (1990 onward to span multiple regimes)
macro_start = '1990-01-01'

gdp = web.DataReader('GDPC1',    'fred', macro_start, END)
cpi = web.DataReader('CPILFESL', 'fred', macro_start, END)
une = web.DataReader('UNRATE',   'fred', macro_start, END)
ten = web.DataReader('DGS10',    'fred', macro_start, END)
rec = web.DataReader('USREC',    'fred', macro_start, END)

print('Shapes:')
print('  GDPC1    (real GDP, quarterly):', gdp.shape)
print('  CPILFESL (core CPI, monthly):  ', cpi.shape)
print('  UNRATE   (unemployment):       ', une.shape)
print('  DGS10    (10y yield, daily):   ', ten.shape)
print('  USREC    (recession 0/1):      ', rec.shape)

In [ ]:
# Transformations: YoY for GDP/CPI, monthly average for 10y yield
gdp_yoy = gdp.pct_change(4).dropna().rename(columns={'GDPC1': 'GDP_YoY'})
cpi_yoy = cpi.pct_change(12).dropna().rename(columns={'CPILFESL': 'CPI_YoY'})
ten_m   = ten.resample('ME').mean().rename(columns={'DGS10': 'TenY'})
une_m   = une.rename(columns={'UNRATE': 'Unemp'})
rec_m   = rec.rename(columns={'USREC': 'Rec'})

# Combine into a monthly macro panel — forward-fill quarterly GDP
macro = (pd.concat([gdp_yoy, cpi_yoy, une_m, ten_m, rec_m], axis=1)
           .resample('ME').last().ffill().dropna())
print(f'Macro panel: {macro.shape[0]} months from {macro.index.min().date()} to {macro.index.max().date()}')
macro.tail()

In [ ]:
# 2x2 panel: visualise all four macro variables with NBER recession shading
fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True)
panels = [
    (axes[0,0], macro['GDP_YoY']*100, 'Real GDP YoY (%)',      NAVY,  2.5),
    (axes[0,1], macro['CPI_YoY']*100, 'Core CPI YoY (%)',      RED,   3.0),
    (axes[1,0], macro['Unemp'],       'Unemployment rate (%)', GOLD,  None),
    (axes[1,1], macro['TenY'],        '10y Treasury yield (%)', GREEN, None),
]
for ax, series, title, color, threshold in panels:
    ax.plot(series.index, series, color=color, lw=1.2)
    if threshold is not None:
        ax.axhline(threshold, color=MUTED, ls='--', lw=0.7,
                   label=f'{threshold}% threshold')
        ax.legend(loc='upper right', fontsize=8)
    for dt, val in rec.itertuples():
        if val == 1:
            ax.axvspan(dt, dt + pd.Timedelta(days=31),
                       color='lightgray', alpha=0.4)
    ax.set_title(title, fontweight='bold', color=INK)
plt.suptitle('US Macro Panel — grey bands = NBER-dated recessions',
             fontweight='bold', color=INK, y=1.02)
plt.tight_layout(); plt.show()

## 5 · Investment Clock — a simple regime model

The **Investment Clock** (Merrill Lynch, 2004) classifies each month into one of four regimes using two macro signals:

| Phase           | Growth | Inflation | Assets that *historically* worked  |
|-----------------|--------|-----------|--------------------------------------|
| **Reflection** | low    | low       | Long-duration bonds, defensives     |
| **Recovery**   | high   | low       | Equities (cyclicals), credit        |
| **Overheat**   | high   | high      | Commodities, real assets            |
| **Stagflation**| low    | high      | Cash, gold, short-duration          |

We use **hard cut-offs** (GDP 2.5%, CPI 3.0%) for transparency. In production you would use rolling z-scores or smoothed trends — here the priority is that every student can trace exactly how a month gets its label.

In [ ]:
def investment_clock(df, gdp_cut=0.025, cpi_cut=0.030):
    """Classify each row into one of four macro regimes."""
    out = df.copy()
    out['Growth']    = np.where(out['GDP_YoY'] > gdp_cut, 'high', 'low')
    out['Inflation'] = np.where(out['CPI_YoY'] > cpi_cut, 'high', 'low')
    conds = [
        (out['Growth']=='low')  & (out['Inflation']=='low'),
        (out['Growth']=='high') & (out['Inflation']=='low'),
        (out['Growth']=='high') & (out['Inflation']=='high'),
        (out['Growth']=='low')  & (out['Inflation']=='high'),
    ]
    out['Regime'] = np.select(
        conds, ['Reflection','Recovery','Overheat','Stagflation'], default='Unknown')
    return out

clk = investment_clock(macro)
regime_colors = {
    'Reflection':  '#9CA3AF',
    'Recovery':    '#2D6E3E',
    'Overheat':    '#D4A017',
    'Stagflation': '#B32D2D',
}
print(f'Most recent regime ({clk.index[-1].date()}): {clk["Regime"].iloc[-1]}')
print('\nRegime frequency (1990-2024):')
print(clk['Regime'].value_counts())

In [ ]:
# Visualise: GDP & CPI YoY coloured by current regime + pie of regime frequency
fig, axes = plt.subplots(1, 2, figsize=(13, 5),
                         gridspec_kw={'width_ratios':[2,1]})

axes[0].plot(clk.index, clk['GDP_YoY']*100, color=NAVY, lw=1.0, label='GDP YoY')
axes[0].plot(clk.index, clk['CPI_YoY']*100, color=RED,  lw=1.0, label='CPI YoY')
ymin, ymax = -5, 12
for phase, col in regime_colors.items():
    mask = (clk['Regime'] == phase).values
    axes[0].fill_between(clk.index, ymin, ymax, where=mask,
                          color=col, alpha=0.18, label=phase)
axes[0].set_ylim(ymin, ymax); axes[0].set_ylabel('YoY (%)')
axes[0].set_title('Investment Clock — phase classification',
                   fontweight='bold', color=INK)
axes[0].legend(loc='upper right', ncol=2, fontsize=8)

counts = clk['Regime'].value_counts()
axes[1].pie(counts.values, labels=counts.index,
             colors=[regime_colors[k] for k in counts.index],
             autopct='%.0f%%', startangle=90,
             wedgeprops={'edgecolor':'white','linewidth':1})
axes[1].set_title('Regime frequency (1990-2024)',
                   fontweight='bold', color=INK)
plt.tight_layout(); plt.show()

## 6 · What worked in each regime?

We compute the **average annualised return** of each ETF, **conditional on the macro regime** active that month. This is not a backtest — it is an historical conditional distribution.

The heatmap doubles as a tactical playbook.

In [ ]:
# Align our monthly returns with the regime panel
monthly_rets = (1 + rets).resample('ME').prod() - 1
clk_m = clk.resample('ME').last()
joined = monthly_rets.join(clk_m['Regime'], how='inner').dropna(subset=['Regime'])

# Average monthly return per regime per asset, annualised (x12)
regime_ret = joined.groupby('Regime').mean(numeric_only=True) * 12
order = ['Reflection','Recovery','Overheat','Stagflation']
regime_ret = regime_ret.reindex([r for r in order if r in regime_ret.index])

# How many months in each regime within OUR sample (2014-2024)?
sample_n = joined['Regime'].value_counts().reindex(order).fillna(0).astype(int)
print('Sample size (months) per regime in 2014-2024:')
print(sample_n.to_string())
print('\n[!] Warning: regimes with <20 months are statistically thin — interpret with care.')

In [ ]:
# Conditional-return heatmap
fig, ax = plt.subplots(figsize=(11, 4))
sns.heatmap(regime_ret*100, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
            cbar_kws={'label':'Annualised return (%)'}, ax=ax,
            linewidths=0.5, linecolor='white')
ax.set_title('Annualised return per asset, conditional on macro regime',
             fontweight='bold', color=INK)
ax.set_xlabel(''); ax.set_ylabel('')
plt.tight_layout(); plt.show()

## 7 · Five simple portfolio baselines

We build five portfolios with **short, readable functions** — no PyPortfolioOpt black boxes, no HRP from scratch.

1. **Equal-Weight (1/N)** — DeMiguel-Garlappi-Uppal (2009) benchmark
2. **Min-Vol** — minimise portfolio variance, long-only, fully invested
3. **Max-Sharpe** — tangency portfolio, long-only, fully invested
4. **Permanent (Browne 1981)** — 25% stocks / 25% long bonds / 25% gold / 25% short bonds
5. **All-Weather (Dalio 1996)** — 30% stocks / 40% long bonds / 15% int. bonds / 7.5% gold / 7.5% commodities

In [ ]:
# Annualised inputs
mu = rets.mean().values * TRADING_DAYS
Sigma = (rets.cov() * TRADING_DAYS).values
n = len(mu)
tickers = list(rets.columns)

def perf(w, mu=mu, Sigma=Sigma, rf=rf_annual):
    """Return (annualised return, vol, Sharpe) for weights w."""
    r = w @ mu
    v = np.sqrt(w @ Sigma @ w)
    s = (r - rf) / v if v > 0 else 0.0
    return r, v, s

def solve_max_sharpe(mu, Sigma, rf=rf_annual):
    n = len(mu)
    cons = ({'type':'eq', 'fun': lambda w: w.sum() - 1.0},)
    bnd  = tuple((0.0, 1.0) for _ in range(n))
    res  = minimize(lambda w: -perf(w, mu, Sigma, rf)[2],
                    np.repeat(1/n, n),
                    method='SLSQP', bounds=bnd, constraints=cons)
    return res.x

def solve_min_vol(Sigma):
    n = Sigma.shape[0]
    cons = ({'type':'eq', 'fun': lambda w: w.sum() - 1.0},)
    bnd  = tuple((0.0, 1.0) for _ in range(n))
    res  = minimize(lambda w: np.sqrt(w @ Sigma @ w),
                    np.repeat(1/n, n),
                    method='SLSQP', bounds=bnd, constraints=cons)
    return res.x

def static_w(tickers, alloc):
    """Build a weight vector from a {ticker: fraction} dict; renormalise."""
    w = np.zeros(len(tickers))
    for t, f in alloc.items():
        if t in tickers:
            w[tickers.index(t)] = f
    return w / w.sum() if w.sum() > 0 else w

w_ew   = np.repeat(1/n, n)
w_mv   = solve_min_vol(Sigma)
w_ms   = solve_max_sharpe(mu, Sigma)
w_perm = static_w(tickers, {'SPY':0.25, 'TLT':0.25, 'GLD':0.25, 'IEF':0.25})
w_aw   = static_w(tickers, {'SPY':0.30, 'TLT':0.40, 'IEF':0.15,
                             'GLD':0.075, 'DBC':0.075})

scorecard = pd.DataFrame({
    'Equal-Wt':    perf(w_ew),
    'Min-Vol':     perf(w_mv),
    'Max-Sharpe':  perf(w_ms),
    'Permanent':   perf(w_perm),
    'All-Weather': perf(w_aw),
}, index=['Ann. Return', 'Ann. Vol', 'Sharpe']).round(3)
scorecard

In [ ]:
# Visualise the five allocations side-by-side
alloc_df = pd.DataFrame({
    'Equal-Wt': w_ew, 'Min-Vol': w_mv, 'Max-Sharpe': w_ms,
    'Permanent': w_perm, 'All-Weather': w_aw,
}, index=tickers)
fig, ax = plt.subplots(figsize=(11, 5))
alloc_df.plot(kind='bar', ax=ax,
              color=[MUTED, GREEN, GOLD, NAVY, RED],
              edgecolor='black', linewidth=0.4)
ax.set_title('Portfolio weights by strategy', fontweight='bold', color=INK)
ax.set_ylabel('Weight')
plt.xticks(rotation=45)
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout(); plt.show()
print('\nObservation: Max-Sharpe is the most concentrated;')
print('Permanent and All-Weather are the most diversified by construction.')

## 8 · Regime-aware tilt — overlay macro information

We take **Max-Sharpe** as a baseline, then tilt the weights toward assets that historically performed well in the **current** regime. Simple, transparent, and easy to inspect.

$$w_i^{\text{tilted}} \propto w_i^{\text{base}} \cdot \bigl( 1 + \theta \cdot z_i^{\text{regime}} \bigr)$$

where $z_i^{\text{regime}}$ is the z-score of asset $i$'s return in the current regime vs the cross-regime mean, and $\theta$ controls the tilt strength.

In [ ]:
def regime_tilt(base_w, regime_table, current_regime, theta=0.30):
    """Tilt baseline weights by per-asset regime z-score."""
    if current_regime not in regime_table.index:
        return base_w
    cur = regime_table.loc[current_regime]
    z = (cur - regime_table.mean(axis=0)) / (regime_table.std(axis=0) + 1e-9)
    z = z.clip(-2, 2)                                  # cap extremes
    mult = (1.0 + theta * z).reindex(rets.columns).fillna(1.0).values
    w = np.maximum(base_w * mult, 0)
    return w / w.sum() if w.sum() > 0 else base_w

current  = clk['Regime'].iloc[-1]
w_tilted = regime_tilt(w_ms, regime_ret, current, theta=0.30)
r_t, v_t, s_t = perf(w_tilted)
r_b, v_b, s_b = perf(w_ms)
print(f'Current regime ({clk.index[-1].date()}): {current}')
print(f'Baseline (Max-Sharpe):  Ann.Ret {r_b:.3f}  Vol {v_b:.3f}  Sharpe {s_b:.3f}')
print(f'Regime-tilted:          Ann.Ret {r_t:.3f}  Vol {v_t:.3f}  Sharpe {s_t:.3f}')

In [ ]:
# Visualise the tilt vs the baseline
cmp = pd.DataFrame({
    'Max-Sharpe (base)':           w_ms,
    f'Regime-tilted ({current})':  w_tilted,
}, index=tickers)
fig, ax = plt.subplots(figsize=(11, 4))
cmp.plot(kind='bar', ax=ax, color=[MUTED, GOLD],
         edgecolor='black', linewidth=0.4)
ax.set_title(f'Regime-aware tilt — active phase: {current}',
             fontweight='bold', color=INK)
ax.set_ylabel('Weight')
plt.xticks(rotation=45)
ax.legend(loc='upper right')
plt.tight_layout(); plt.show()
print('\nInterpretation: assets get scaled by their historical performance in the')
print('CURRENT regime, not by their full-sample mean. A simple form of conditional allocation.')

## 9 · Caveats — what would break this in production

This notebook is **didactic** by design. Before anyone deploys real capital based on similar logic, they would have to address:

1. **Look-ahead bias.** We classified history retrospectively. In real time, GDP is published with a ~30-day lag, and revisions can change a regime classification *after the fact*.
2. **In-sample optimism.** Max-Sharpe weights were fitted on the same data we evaluated them on. A walk-forward backtest with transaction costs typically halves the realised Sharpe.
3. **Sample size.** With ~30 years of macro data, we have only a handful of full regime cycles. Conditional returns based on <20 months should be treated as exploratory.
4. **Hard thresholds hide gradients.** GDP at 2.4% and 0.5% are very different, yet both get labelled 'low' here.
5. **Non-stationarity.** The equity-bond correlation flipped in 2022; risk premia move with monetary policy; macro relationships are not laws of nature.
6. **Frictions.** Transaction costs, slippage, taxes, borrow constraints all eat returns. None are modelled here.

Used as a **complement** to factor models and risk management — not a substitute — macro regime analysis adds interpretable, narrative-friendly context to a quantitative portfolio.

---

## 10 · Discussion & references

### What we learned

1. Daily returns are **fat-tailed** for every asset — Gaussian assumptions fail.
2. Equity-bond **correlations are not constant** — 2022 reminded everyone.
3. **Macro variables matter**: GDP/CPI YoY transformations and recession overlays give context that pure return-based models lack.
4. **Static portfolios** (Permanent, All-Weather) are surprisingly hard to beat once you account for trading costs and behavioural drift.
5. **Regime tilts** are intuitive and easy to explain — that is both their strength (interpretability) and their weakness (over-simplification).

### Where to go next

- Implement a **walk-forward backtest** with realistic frictions.
- Replace hard thresholds with **rolling z-scores** for the macro signals.
- Add a **factor exposure analysis** (Fama-French 3-factor) on the tilted portfolio.
- Stress-test on the **COVID shock** (Feb-Apr 2020) or the **2022 bond rout**.

### References

- Markowitz, H. (1952). *Portfolio Selection*. Journal of Finance.
- Browne, H. (1981). *Inflation-Proofing Your Investments*.
- Dalio, R. (1996). *Engineering Targeted Returns and Risks*. Bridgewater.
- DeMiguel, V., Garlappi, L., Uppal, R. (2009). *Optimal Versus Naive Diversification: How Inefficient is the 1/N Portfolio Strategy?*. RFS.
- Greetham, T., Hartnett, M. (2004). *The Investment Clock*. Merrill Lynch.
- Balasso, P. *YoutubeTrading* — [github.com/paolobalasso/YoutubeTrading](https://github.com/paolobalasso/YoutubeTrading).

---

**End of notebook.** ~30 seconds of runtime on Colab CPU. The goal was *content sedimentation*, not deployment — every cell should be inspectable, every plot tells a story.